In [1]:
from datasets import load_dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer, TextClassificationPipeline
import torch


### Load tained model

In [2]:
model = AutoModelForSequenceClassification.from_pretrained('bert-base-cased')

Downloading:   0%|          | 0.00/433 [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/436M [00:00<?, ?B/s]

checkpoint before trying to load file. File's name is: /home/sago/.cache/torch/transformers/d8f11f061e407be64c4d5d7867ee61d1465263e24085cfa26abf183fdc830569.3fadbea36527ae472139fe84cddaa65454d7429f12d543d80bfc3ad70de55ac2


Some weights of the model checkpoint at bert-base-cased were not used when initializing BertForSequenceClassification: ['cls.predictions.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.decoder.weight', 'cls.seq_relationship.weight', 'cls.seq_relationship.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.LayerNorm.bias']
- This IS expected if you are initializing BertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPretraining model).
- This IS NOT expected if you are initializing BertForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at b

In [5]:
model.load_state_dict(torch.load('state.pt'), strict=False)

checkpoint before trying to load file. File's name is: state.pt


_IncompatibleKeys(missing_keys=[], unexpected_keys=['bert.embeddings.position_ids'])

In [7]:
model = model.eval()

### load and Tokenize MRPC test

In [8]:
tokenizer = AutoTokenizer.from_pretrained('bert-base-cased')

dataset_test = load_dataset('glue', 'mrpc', split='test')

#Encode dataset
def encode(examples):
    return tokenizer(examples['sentence1'], examples['sentence2'], truncation=True, padding='max_length')

dataset_test = dataset_test.map(encode, batched=True)
#Rename label to labels
dataset_test = dataset_test.map(lambda rows: {'labels': rows['label']}, batched=True)

#Set dataset format and dataloader
dataset_test.set_format(type='torch', columns=['input_ids', 'token_type_ids', 'attention_mask', 'labels'])
test_loader = torch.utils.data.DataLoader(dataset_test, batch_size=16)

Downloading:   0%|          | 0.00/213k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3668 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/408 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1725 [00:00<?, ? examples/s]

Dataset glue downloaded and prepared to /home/sago/.cache/huggingface/datasets/glue/mrpc/1.0.0/dacbe3125aa31d7f70367a07a8a9e72a5a0bfeb5fc42e75c9db75b96da6053ad. Subsequent calls will reuse this data.


Map:   0%|          | 0/1725 [00:00<?, ? examples/s]

Map:   0%|          | 0/1725 [00:00<?, ? examples/s]

In [9]:
results = []
for i, batch in enumerate(test_loader):
    _, outputs = model(**batch)
    outputs = torch.max(outputs, 1).indices.numpy()
    results.extend(outputs)


: 

: 

In [9]:
# Calculate accuracy
correct_predictions = 0
total_predictions = len(dataset_test)
for predicted, example in zip(results, dataset_test):
    predicted_label = predicted
    ground_truth_label = example['labels']
    if predicted_label == ground_truth_label:
        correct_predictions += 1

accuracy = correct_predictions / total_predictions

print(f"Accuracy: {accuracy * 100:.2f}%")

Accuracy: 80.17%
